In [1]:

!pip -q install -U langchain langchain-groq langchain-core gspread


In [31]:

import os
from google.colab import userdata
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = userdata.get("AgenticAI")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_retries=3
)

print("Groq GPT-OSS 120B initialized successfully!")


Groq GPT-OSS 120B initialized successfully!


In [32]:

auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

try:
    sh = gc.open("Sports ShopMate AI Orders")
except gspread.SpreadsheetNotFound:
    sh = gc.create("Sports ShopMate AI Orders")

ws = sh.sheet1

if not ws.get_all_values():
    ws.append_row([
        "Customer Name",
        "Product",
        "Category",
        "Quantity",
        "Size / Specification",
        "Color",
        "Order Date",
        "Total Amount (Rs)"
    ])

print("Google Sheet ready!")
print(sh.url)


Google Sheet ready!
https://docs.google.com/spreadsheets/d/1BjtDlRJkAIhjW_0sYb8LCKMQGWBXPm17g0joFTAvOb0


In [33]:

PRODUCTS = {
    "cricket bat": {
        "category": "Cricket",
        "price": 1200,
        "colors": ["Natural Wood"],
        "sizes": ["SH", "Harrow", "Full Size"],
        "details": "English willow style practice bat",
        "stock": 15
    },

    "cricket ball": {
        "category": "Cricket",
        "price": 250,
        "colors": ["Red", "White"],
        "sizes": ["Standard"],
        "details": "Leather cricket ball",
        "stock": 40
    },

    "cricket helmet": {
        "category": "Cricket",
        "price": 1500,
        "colors": ["Black", "Blue", "Navy"],
        "sizes": ["S", "M", "L"],
        "details": "Protective cricket helmet",
        "stock": 10
    },

    "cricket gloves": {
        "category": "Cricket",
        "price": 850,
        "colors": ["White", "Blue"],
        "sizes": ["S", "M", "L"],
        "details": "Batting gloves",
        "stock": 20
    },

    "football": {
        "category": "Football",
        "price": 700,
        "colors": ["White", "Yellow", "Blue"],
        "sizes": ["3", "4", "5"],
        "details": "Training football",
        "stock": 25
    },

    "football shoes": {
        "category": "Football",
        "price": 1800,
        "colors": ["Black", "Blue", "White"],
        "sizes": ["6", "7", "8", "9", "10"],
        "details": "Football studs",
        "stock": 15
    },

    "goalkeeper gloves": {
        "category": "Football",
        "price": 950,
        "colors": ["Black", "Blue", "Green"],
        "sizes": ["S", "M", "L"],
        "details": "Goalkeeper gloves",
        "stock": 12
    },

    "badminton racket": {
        "category": "Badminton",
        "price": 1100,
        "colors": ["Black", "Red", "Blue"],
        "sizes": ["Standard"],
        "details": "Lightweight badminton racket",
        "stock": 20
    },

    "shuttlecock": {
        "category": "Badminton",
        "price": 300,
        "colors": ["White"],
        "sizes": ["Standard"],
        "details": "Pack of 6 shuttlecocks",
        "stock": 30
    },

    "badminton shoes": {
        "category": "Badminton",
        "price": 1600,
        "colors": ["White", "Black", "Blue"],
        "sizes": ["6", "7", "8", "9", "10"],
        "details": "Non-marking court shoes",
        "stock": 12
    },

    "tennis racket": {
        "category": "Tennis",
        "price": 2200,
        "colors": ["Black", "Red", "Blue"],
        "sizes": ["Standard"],
        "details": "Tennis racket for training",
        "stock": 10
    },

    "tennis ball": {
        "category": "Tennis",
        "price": 150,
        "colors": ["Yellow"],
        "sizes": ["Standard"],
        "details": "Tennis practice ball",
        "stock": 50
    },

    "basketball": {
        "category": "Basketball",
        "price": 900,
        "colors": ["Orange", "Brown"],
        "sizes": ["5", "6", "7"],
        "details": "Rubber basketball",
        "stock": 20
    },

    "volleyball": {
        "category": "Volleyball",
        "price": 750,
        "colors": ["White", "Yellow", "Blue"],
        "sizes": ["Standard"],
        "details": "Training volleyball",
        "stock": 15
    },

    "table tennis bat": {
        "category": "Table Tennis",
        "price": 500,
        "colors": ["Black", "Red"],
        "sizes": ["Standard"],
        "details": "Table tennis paddle",
        "stock": 25
    },

    "table tennis ball": {
        "category": "Table Tennis",
        "price": 100,
        "colors": ["White", "Orange"],
        "sizes": ["Standard"],
        "details": "Pack of 3 balls",
        "stock": 40
    },

    "dumbbells": {
        "category": "Fitness",
        "price": 600,
        "colors": ["Black"],
        "sizes": ["2kg", "5kg", "10kg"],
        "details": "Pair of fitness dumbbells",
        "stock": 20
    },

    "yoga mat": {
        "category": "Fitness",
        "price": 450,
        "colors": ["Purple", "Blue", "Pink", "Black"],
        "sizes": ["Standard"],
        "details": "Non-slip exercise yoga mat",
        "stock": 30
    },

    "resistance bands": {
        "category": "Fitness",
        "price": 350,
        "colors": ["Red", "Green", "Blue"],
        "sizes": ["Light", "Medium", "Heavy"],
        "details": "Resistance training bands",
        "stock": 25
    },

    "treadmill": {
        "category": "Gym Equipment",
        "price": 25000,
        "colors": ["Black"],
        "sizes": ["Standard"],
        "details": "Motorized home treadmill",
        "stock": 5
    },

    "skipping rope": {
        "category": "Fitness",
        "price": 200,
        "colors": ["Black", "Blue", "Red"],
        "sizes": ["Standard"],
        "details": "Adjustable skipping rope",
        "stock": 35
    },

    "athletics shoes": {
        "category": "Athletics",
        "price": 2000,
        "colors": ["Black", "Blue", "White"],
        "sizes": ["6", "7", "8", "9", "10"],
        "details": "Running and training shoes",
        "stock": 12
    }
}

print("Total products:", len(PRODUCTS))


Total products: 22


In [34]:

from pydantic import BaseModel, Field

class Order(BaseModel):
    name: str = Field(description="Customer name")

    item: str = Field(
        description="Exact product name from the catalog"
    )

    qty: int = Field(
        description="Number of products",
        ge=1
    )

    size: str = Field(
        description="Product size or specification"
    )

    color: str = Field(
        description="Selected product colour"
    )

    date: str = Field(
        description="Delivery or pickup date"
    )


In [35]:

from langchain.tools import tool

@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the price and total cost of a sports product."""

    product = PRODUCTS.get(item.lower().strip())

    if not product:
        return f"{item} is unavailable in our catalog."

    if qty < 1:
        return "Quantity must be at least 1."

    if qty > product["stock"]:
        return f"Only {product['stock']} units are currently available."

    total = product["price"] * qty

    return (
        f"Product: {item}\n"
        f"Unit Price: Rs {product['price']}\n"
        f"Quantity: {qty}\n"
        f"Total: Rs {total}"
    )


In [36]:

@tool
def check_product(item: str) -> str:
    """Check sports product category, price, sizes, colours and stock."""

    product = PRODUCTS.get(item.lower().strip())

    if not product:
        return (
            f"{item} is unavailable. "
            f"Available products: {', '.join(PRODUCTS.keys())}"
        )

    return f"""
Product: {item}
Category: {product['category']}
Price: Rs {product['price']}
Colours: {', '.join(product['colors'])}
Sizes / Specifications: {', '.join(product['sizes'])}
Details: {product['details']}
Available Stock: {product['stock']}
"""


In [37]:

@tool
def recommend_products(budget: int) -> str:
    """Recommend sports equipment within the customer's budget."""

    if budget < 0:
        return "Please provide a valid budget."

    matches = []

    for item, details in PRODUCTS.items():
        if details["price"] <= budget and details["stock"] > 0:
            matches.append(
                f"{item} - Rs {details['price']} "
                f"({details['category']})"
            )

    if not matches:
        return "No products are available within this budget."

    return (
        f"Sports products within Rs {budget}:\n"
        + "\n".join(matches)
    )



In [38]:

@tool
def log_order(
    name: str,
    item: str,
    qty: int,
    size: str,
    color: str,
    date: str
) -> str:
    """Save a customer order to Google Sheets.
    Call only after explicit customer confirmation."""

    item = item.lower().strip()

    try:
        order = Order(
            name=name.strip(),
            item=item,
            qty=qty,
            size=size.strip(),
            color=color.strip().title(),
            date=date.strip()
        )

    except Exception as e:
        return f"Could not validate order: {e}"

    product = PRODUCTS.get(order.item)

    if product is None:
        return "Product not found."

    if order.qty > product["stock"]:
        return f"Only {product['stock']} units are available."

    if order.color.lower() not in [
        c.lower() for c in product["colors"]
    ]:
        return f"Colour {order.color} is unavailable."

    if order.size.lower() not in [
        s.lower() for s in product["sizes"]
    ]:
        return f"Size or specification {order.size} is unavailable."

    total = product["price"] * order.qty

    try:
        ws.append_row([
            order.name,
            order.item,
            product["category"],
            order.qty,
            order.size,
            order.color,
            order.date,
            total
        ])

        return f"""
ORDER SAVED SUCCESSFULLY!

Customer: {order.name}
Product: {order.item}
Category: {product['category']}
Quantity: {order.qty}
Size / Specification: {order.size}
Colour: {order.color}
Order Date: {order.date}
Total Amount: Rs {total}
"""

    except Exception as e:
        return f"Google Sheets error: {e}"


In [39]:

from langchain.agents import create_agent

SYSTEM = """
ROLE:
You are Sports ShopMate AI, a friendly sports equipment
shopping assistant.

BUSINESS:
You help customers discover sports equipment,
compare prices, check availability and place orders.

PRODUCT CATALOG:
Use the provided PRODUCTS catalog and tools.
Never invent products, prices, sizes, colours or stock.

TOOLS:
1. Use check_product to check product information.
2. Use get_price to calculate prices.
3. Use recommend_products for budget-based recommendations.
4. Use log_order to save confirmed orders.

ORDER RULES:
1. Collect customer name.
2. Collect product name.
3. Collect quantity.
4. Collect size or specification.
5. Collect colour.
6. Collect delivery or pickup date.
7. Check product availability before accepting an order.
8. Calculate the total price using get_price.
9. Summarize the complete order.
10. Ask the customer to confirm the order.
11. Call log_order only after explicit confirmation.
12. Never claim an order is saved unless the tool succeeds.
13. Never invent stock availability.

LANGUAGE:
Support both English and Telugu.
Respond in the same language used by the customer.
Use simple Telugu when the customer speaks Telugu.

CONVERSATION:
Remember the details shared during the current conversation.
Do not repeatedly ask for information already provided.

FORMAT:
Keep responses short, friendly and professional.
Use clear headings and bullet points when needed.
"""

agent = create_agent(
    model=llm,
    tools=[
        get_price,
        check_product,
        recommend_products,
        log_order
    ],
    system_prompt=SYSTEM
)

print("Sports ShopMate AI Agent created successfully!")


Sports ShopMate AI Agent created successfully!


In [40]:

history = []

def chat(text: str) -> str:

    global history

    try:
        result = agent.invoke({
            "messages": history[-6:] + [
                {"role": "user", "content": text}
            ]
        })

        history = result["messages"]

        return history[-1].content

    except Exception as e:
        print("Agent error:", e)

        return (
            "Sorry, something went wrong. "
            "Please try again."
        )


In [41]:

print(chat("Hi! What sports equipment do you have?"))


**Hello! 👋**  

We have a wide selection of sports equipment, including:

- **Balls** – football, basketball, cricket, tennis, volleyball, etc.  
- **Footwear** – running shoes, football boots, indoor trainers.  
- **Apparel** – jerseys, shorts, tracksuits, compression wear.  
- **Rackets & Bats** – tennis, badminton, cricket, baseball.  
- **Fitness gear** – dumbbells, yoga mats, resistance bands, kettlebells.  
- **Outdoor gear** – camping tents, hiking boots, cycling helmets.  

What type of sport or specific item are you looking for? Feel free to tell me the sport, any brand you prefer, or a budget, and I’ll help you find the perfect gear.


In [42]:

print(chat("What is the price of 2 cricket bats?"))


**Cricket Bats – Pricing**

- **Item:** Cricket Bat  
- **Unit Price:** Rs 1,200  
- **Quantity:** 2  
- **Total Cost:** Rs 2,400  

Let me know if you’d like to choose a size (SH, Harrow, Full Size) and a colour (Natural Wood), or if you’d like to place the order. 😊


In [43]:

print(chat("Do you have badminton rackets? What colours are available?"))


**Badminton Racket**

- **Price:** Rs 1,100 each  
- **Available Colours:** Black, Red, Blue  
- **Size/Specification:** Standard  
- **Stock:** 20 units  

Would you like to know more details, see price for a quantity, or place an order? Let me know!


Agent error: Error code: 400 - {'error': {'message': 'failed to template request: failed to render tokenized output: failed to render tokens with harmony: HarmonyError: EncodingError: Message=render failed: Tools should have a name!', 'type': 'invalid_request_error'}}
Sorry, something went wrong. Please try again.
